In [1]:

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("Housing.csv")

X = df.drop("price", axis=1)
y = df["price"]

numeric = X.select_dtypes(include="number").columns
categorical = X.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42
    ),
    "AdaBoost": AdaBoostRegressor(
        n_estimators=100, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, random_state=42
    )
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = -cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    start = time.time()
    pipeline.fit(X_train, y_train)
    training_time = time.time() - start

    predictions = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append([
        name,
        scores.mean(),
        scores.std(),
        rmse,
        r2,
        training_time
    ])

    print("\n", name)
    print("CV RMSE Mean:", round(scores.mean(), 2))
    print("CV RMSE Std:", round(scores.std(), 2))
    print("Test RMSE:", round(rmse, 2))
    print("Test R2 Score:", round(r2, 4))
    print("Training Time:", round(training_time, 4), "seconds")

    feature_names = (
        pipeline.named_steps["preprocessor"]
        .get_feature_names_out()
    )

    importance = pipeline.named_steps["model"].feature_importances_

    feature_importance = pd.Series(
        importance, index=feature_names
    ).sort_values(ascending=False)

    print("\nTop 10 Important Features:")
    print(feature_importance.head(10))

results_df = pd.DataFrame(
    results,
    columns=[
        "Model", "CV RMSE Mean", "CV RMSE Std",
        "Test RMSE", "R2 Score", "Training Time"
    ]
)

print("\nModel Comparison:")
print(results_df.to_string(index=False))



 Decision Tree
CV RMSE Mean: 1523288.35
CV RMSE Std: 206229.07
Test RMSE: 1699626.14
Test R2 Score: 0.4285
Training Time: 0.015 seconds

Top 10 Important Features:
num__area                            0.482894
num__bathrooms                       0.176391
num__stories                         0.054809
num__bedrooms                        0.048320
cat__airconditioning_no              0.039067
num__parking                         0.038260
cat__furnishingstatus_unfurnished    0.033526
cat__prefarea_no                     0.020503
cat__basement_yes                    0.016287
cat__hotwaterheating_no              0.014050
dtype: float64

 Random Forest
CV RMSE Mean: 1084959.97
CV RMSE Std: 76512.2
Test RMSE: 1398658.02
Test R2 Score: 0.613
Training Time: 0.5389 seconds

Top 10 Important Features:
num__area                            0.464028
num__bathrooms                       0.152546
num__parking                         0.055606
num__stories                         0.054816
num__bedrooms